In [ ]:
import pandas as pd



In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Phase 1 – Task 1.1: Load data and inspect columns

import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 200)


PICKLE_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl")

assert PICKLE_PATH.exists(), f"File not found at: {PICKLE_PATH}. Please check the path."

df = pd.read_pickle(PICKLE_PATH)


print(f"Loaded: {PICKLE_PATH.name}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# Quick info about columns and types
df.info()


Loaded: df_sub.pkl
Shape: 679,968 rows × 81 columns

<class 'pandas.core.frame.DataFrame'>
Index: 679968 entries, 0 to 679969
Data columns (total 81 columns):
 #   Column                                     Non-Null Count   Dtype         
---  ------                                     --------------   -----         
 0   Job Filing Number                          679968 non-null  object        
 1   Filing Status                              679968 non-null  object        
 2   House No                                   679968 non-null  object        
 3   Street Name                                679968 non-null  object        
 4   Borough                                    679968 non-null  object        
 5   Block                                      679968 non-null  int64         
 6   LOT                                        679968 non-null  int64         
 7   Bin                                        679968 non-null  int64         
 8   Commmunity - Board                  

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


!pip install -q umap-learn
import umap


OUTPUT_DIR = Path("/content/drive/MyDrive/Attempt3-Assignment4-CS424")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("df shape:", df.shape)


Output directory: /content/drive/MyDrive/Attempt3-Assignment4-CS424
df shape: (679968, 81)


In [ ]:


# Core columns we definitely want to keep
base_cols = [
    "Job Filing Number",
    "Filing Date",
    "Approved Date",
    "Job Type",
    "Borough",
    "NTA",
    "Initial Cost",
    "FloorArea_imputed",
    "Existing Dwelling Units",
    "Proposed Dwelling Units",
    "Existing Stories",
    "Proposed No of Stories",
    "Existing Height",
    "Proposed Height",
    "med_boro",
    "med_boro_job",
    "Latitude",
    "Longitude",
    "Building Type",
]

# All boolean / boolean-like columns (work types, flags)
bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns.tolist()
print("Boolean columns found:", bool_cols)

# Combine base + bool columns
keep_cols = list(dict.fromkeys(base_cols + bool_cols))

df_emb = df[keep_cols].copy()
print("df_emb shape:", df_emb.shape)
df_emb.head()


Boolean columns found: ['Sprinkler (Work Type)', 'Plumbing (Work Type)', 'Standpipe', 'Antenna', 'Curb Cut', 'Sign', 'Fence', 'Scaffold', 'Shed', 'Boiler Equipment (Work Type)', 'Earth Work (Work Type)', 'Foundation (Work Type)', 'General Construction (Work Type)', 'Mechanical Systems (Work Type)', 'Place of Assembly (Work Type)', 'Protection Mechanical Methods (Work Type)', 'Sidewalk Shed (Work Type)', 'Structural (Work Type)', 'Temporary Place of Assembly (Work Type)', 'FloorArea_was_imputed']
df_emb shape: (679968, 39)


,Job Filing Number,Filing Date,Approved Date,Job Type,Borough,NTA,Initial Cost,FloorArea_imputed,Existing Dwelling Units,Proposed Dwelling Units,Existing Stories,Proposed No of Stories,Existing Height,Proposed Height,med_boro,med_boro_job,Latitude,Longitude,Building Type,Sprinkler (Work Type),Plumbing (Work Type),Standpipe,Antenna,Curb Cut,Sign,Fence,Scaffold,Shed,Boiler Equipment (Work Type),Earth Work (Work Type),Foundation (Work Type),General Construction (Work Type),Mechanical Systems (Work Type),Place of Assembly (Work Type),Protection Mechanical Methods (Work Type),Sidewalk Shed (Work Type),Structural (Work Type),Temporary Place of Assembly (Work Type),FloorArea_was_imputed
0,B01237951-S1,2025-08-28,2025-08-29 11:34:34,Alteration,BROOKLYN,Sunset Park (West),24000.0,600.0,0.0,0.0,2.0,2.0,40.0,40.0,2000.0,1435.0,40.661309,-74.000422,Other,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False
1,B01237951-I1,2025-06-20,2025-06-23 15:58:48,Alteration,BROOKLYN,Sunset Park (West),68000.0,1700.0,0.0,0.0,2.0,2.0,40.0,40.0,2000.0,1435.0,40.661309,-74.000422,Other,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
2,B01258172-I1,2025-10-12,NaT,Full Demolition,BROOKLYN,Bedford-Stuyvesant (East),100000.0,2850.0,2.0,NaN,2.0,NaN,35.0,NaN,2000.0,2213.0,40.685604,-73.926614,2 Family,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,Q01195664-S2,2025-10-12,NaT,Alteration,QUEENS,South Richmond Hill,0.0,625.0,1.0,1.0,2.0,2.0,23.0,23.0,1611.6,1075.0,40.688417,-73.829978,1 Family,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
4,M01138173-P2,2025-10-12,NaT,Alteration,MANHATTAN,Midtown-Times Square,1000.0,1120.0,0.0,0.0,NaN,NaN,NaN,NaN,1500.0,1460.0,40.763527,-73.975279,Other,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False


In [ ]:
df_emb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 679968 entries, 0 to 679969
Data columns (total 39 columns):
 #   Column                                     Non-Null Count   Dtype         
---  ------                                     --------------   -----         
 0   Job Filing Number                          679968 non-null  object        
 1   Filing Date                                679968 non-null  datetime64[ns]
 2   Approved Date                              597417 non-null  datetime64[ns]
 3   Job Type                                   679968 non-null  object        
 4   Borough                                    679968 non-null  object        
 5   NTA                                        679968 non-null  object        
 6   Initial Cost                               679968 non-null  float64       
 7   FloorArea_imputed                          679968 non-null  float64       
 8   Existing Dwelling Units                    596441 non-null  float64       
 9   Proposed 

In [ ]:
# Cell 3 – Feature engineering for embeddings

import numpy as np

# 1) Time-based fields from Filing Date
df_emb["year"] = df_emb["Filing Date"].dt.year
df_emb["month"] = df_emb["Filing Date"].dt.month

# Cyclical encoding for month
df_emb["month_sin"] = np.sin(2 * np.pi * df_emb["month"] / 12.0)
df_emb["month_cos"] = np.cos(2 * np.pi * df_emb["month"] / 12.0)

# Month start + mmYYYY string (for later Vega-Lite time-series)
df_emb["month_start"] = df_emb["Filing Date"].values.astype("datetime64[M]")
df_emb["mmYYYY"] = df_emb["month_start"].dt.strftime("%m/%Y")

# 2) Approval days and capped version
approval_days = (df_emb["Approved Date"] - df_emb["Filing Date"]).dt.days
df_emb["approval_days"] = approval_days
df_emb["approval_days_cap"] = df_emb["approval_days"].clip(lower=0, upper=730)

# Presence flag: has real Approved Date vs NaT
df_emb["has_approval_date"] = df_emb["Approved Date"].notna().astype(int)

# 3) Housing impact: delta dwelling units
df_emb["delta_units"] = (
    df_emb["Proposed Dwelling Units"] - df_emb["Existing Dwelling Units"]
)

# Presence flags for dwelling units
df_emb["has_existing_du"] = df_emb["Existing Dwelling Units"].notna().astype(int)
df_emb["has_proposed_du"] = df_emb["Proposed Dwelling Units"].notna().astype(int)

# 4) Log transforms for skewed scale features
df_emb["log_initial_cost"] = np.log1p(df_emb["Initial Cost"])
df_emb["log_floor_area"] = np.log1p(df_emb["FloorArea_imputed"])

# 5) Coarse NTA: keep top 40 NTAs, rest => "Other NTA"
top_ntas = df_emb["NTA"].value_counts().head(40).index
df_emb["NTA_coarse"] = np.where(df_emb["NTA"].isin(top_ntas), df_emb["NTA"], "Other NTA")

# 6) Handle categorical missing values
df_emb["Building Type"] = df_emb["Building Type"].fillna("Unknown")
# Job Type and Borough have no nulls but safe to be explicit:
df_emb["Job Type"] = df_emb["Job Type"].fillna("Unknown")
df_emb["Borough"] = df_emb["Borough"].fillna("Unknown")

# 7) Convert boolean columns to 0/1 integers
bool_cols = df_emb.select_dtypes(include=["bool", "boolean"]).columns.tolist()
df_emb[bool_cols] = df_emb[bool_cols].astype(int)

print("Top NTAs used in NTA_coarse:", list(top_ntas)[:10], "...")
print("Boolean columns converted to int:", bool_cols)
print("df_emb shape after feature engineering:", df_emb.shape)

df_emb[[
    "Job Filing Number", "Filing Date", "Approved Date", "approval_days_cap",
    "has_approval_date", "Existing Dwelling Units", "Proposed Dwelling Units",
    "has_existing_du", "has_proposed_du",
    "delta_units", "log_initial_cost", "log_floor_area",
    "NTA", "NTA_coarse", "Building Type", "year", "month",
    "month_sin", "month_cos"
]].head()


Top NTAs used in NTA_coarse: ['Midtown-Times Square', 'Upper East Side-Carnegie Hill', 'Midtown South-Flatiron-Union Square', 'Chelsea-Hudson Yards', 'East Midtown-Turtle Bay', 'Upper West Side (Central)', 'Financial District-Battery Park City', 'SoHo-Little Italy-Hudson Square', 'Carroll Gardens-Cobble Hill-Gowanus-Red Hook', 'West Village'] ...
Boolean columns converted to int: ['Sprinkler (Work Type)', 'Plumbing (Work Type)', 'Standpipe', 'Antenna', 'Curb Cut', 'Sign', 'Fence', 'Scaffold', 'Shed', 'Boiler Equipment (Work Type)', 'Earth Work (Work Type)', 'Foundation (Work Type)', 'General Construction (Work Type)', 'Mechanical Systems (Work Type)', 'Place of Assembly (Work Type)', 'Protection Mechanical Methods (Work Type)', 'Sidewalk Shed (Work Type)', 'Structural (Work Type)', 'Temporary Place of Assembly (Work Type)', 'FloorArea_was_imputed']
df_emb shape after feature engineering: (679968, 54)


,Job Filing Number,Filing Date,Approved Date,approval_days_cap,has_approval_date,Existing Dwelling Units,Proposed Dwelling Units,has_existing_du,has_proposed_du,delta_units,log_initial_cost,log_floor_area,NTA,NTA_coarse,Building Type,year,month,month_sin,month_cos
0,B01237951-S1,2025-08-28,2025-08-29 11:34:34,1.0,1,0.0,0.0,1,1,0.0,10.085851,6.398595,Sunset Park (West),Sunset Park (West),Other,2025,8,-8.660254e-01,-0.5
1,B01237951-I1,2025-06-20,2025-06-23 15:58:48,3.0,1,0.0,0.0,1,1,0.0,11.127278,7.438972,Sunset Park (West),Sunset Park (West),Other,2025,6,1.224647e-16,-1.0
2,B01258172-I1,2025-10-12,NaT,NaN,0,2.0,NaN,1,0,NaN,11.512935,7.955425,Bedford-Stuyvesant (East),Bedford-Stuyvesant (East),2 Family,2025,10,-8.660254e-01,0.5
3,Q01195664-S2,2025-10-12,NaT,NaN,0,1.0,1.0,1,1,0.0,0.000000,6.439350,South Richmond Hill,Other NTA,1 Family,2025,10,-8.660254e-01,0.5
4,M01138173-P2,2025-10-12,NaT,NaN,0,0.0,0.0,1,1,0.0,6.908755,7.021976,Midtown-Times Square,Midtown-Times Square,Other,2025,10,-8.660254e-01,0.5


In [ ]:
df_emb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 679968 entries, 0 to 679969
Data columns (total 54 columns):
 #   Column                                     Non-Null Count   Dtype         
---  ------                                     --------------   -----         
 0   Job Filing Number                          679968 non-null  object        
 1   Filing Date                                679968 non-null  datetime64[ns]
 2   Approved Date                              597417 non-null  datetime64[ns]
 3   Job Type                                   679968 non-null  object        
 4   Borough                                    679968 non-null  object        
 5   NTA                                        679968 non-null  object        
 6   Initial Cost                               679968 non-null  float64       
 7   FloorArea_imputed                          679968 non-null  float64       
 8   Existing Dwelling Units                    596441 non-null  float64       
 9   Proposed 

In [ ]:
# Cell 4 – Define numeric & categorical features and preprocessing

# 1) Numeric features for the embedding (Iteration – no stories/heights)
numeric_base = [
    "log_initial_cost",
    "log_floor_area",
    "delta_units",
    "Existing Dwelling Units",
    "Proposed Dwelling Units",
    "approval_days_cap",
    "med_boro",
    "med_boro_job",
    "Latitude",
    "Longitude",
    "year",
    "month_sin",
    "month_cos",
    "has_approval_date",
    "has_existing_du",
    "has_proposed_du",
]

# All 0/1 work-type / flag columns as numeric too
bool_cols = [
    "Sprinkler (Work Type)",
    "Plumbing (Work Type)",
    "Standpipe",
    "Antenna",
    "Curb Cut",
    "Sign",
    "Fence",
    "Scaffold",
    "Shed",
    "Boiler Equipment (Work Type)",
    "Earth Work (Work Type)",
    "Foundation (Work Type)",
    "General Construction (Work Type)",
    "Mechanical Systems (Work Type)",
    "Place of Assembly (Work Type)",
    "Protection Mechanical Methods (Work Type)",
    "Sidewalk Shed (Work Type)",
    "Structural (Work Type)",
    "Temporary Place of Assembly (Work Type)",
    "FloorArea_was_imputed",
]

numeric_features = numeric_base + bool_cols

# 2) Categorical features for one-hot encoding
categorical_features = [
    "Job Type",
    "Borough",
    "Building Type",
    "NTA_coarse",
]

print("Number of numeric features:", len(numeric_features))
print("Numeric features:", numeric_features)
print("\nNumber of categorical features:", len(categorical_features))
print("Categorical features:", categorical_features)

# 3) Define transformers
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Recreating the categorical transformer with the right argument
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False  # dense numpy array
)

# Recreating the ColumnTransformer using the numeric_features & categorical_features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Preprocessor rebuilt successfully.")

Number of numeric features: 36
Numeric features: ['log_initial_cost', 'log_floor_area', 'delta_units', 'Existing Dwelling Units', 'Proposed Dwelling Units', 'approval_days_cap', 'med_boro', 'med_boro_job', 'Latitude', 'Longitude', 'year', 'month_sin', 'month_cos', 'has_approval_date', 'has_existing_du', 'has_proposed_du', 'Sprinkler (Work Type)', 'Plumbing (Work Type)', 'Standpipe', 'Antenna', 'Curb Cut', 'Sign', 'Fence', 'Scaffold', 'Shed', 'Boiler Equipment (Work Type)', 'Earth Work (Work Type)', 'Foundation (Work Type)', 'General Construction (Work Type)', 'Mechanical Systems (Work Type)', 'Place of Assembly (Work Type)', 'Protection Mechanical Methods (Work Type)', 'Sidewalk Shed (Work Type)', 'Structural (Work Type)', 'Temporary Place of Assembly (Work Type)', 'FloorArea_was_imputed']

Number of categorical features: 4
Categorical features: ['Job Type', 'Borough', 'Building Type', 'NTA_coarse']
Preprocessor rebuilt successfully.


In [ ]:
# Cell 5 – Fit the preprocessor and transform df_emb into X_array

X_array = preprocessor.fit_transform(df_emb)

print("X_array shape:", X_array.shape)


feature_names = preprocessor.get_feature_names_out()
print("Number of transformed features:", len(feature_names))
print("First 10 feature names:", feature_names[:10])


X_array shape: (679968, 93)
Number of transformed features: 93
First 10 feature names: ['num__log_initial_cost' 'num__log_floor_area' 'num__delta_units'
 'num__Existing Dwelling Units' 'num__Proposed Dwelling Units'
 'num__approval_days_cap' 'num__med_boro' 'num__med_boro_job'
 'num__Latitude' 'num__Longitude']


In [ ]:
# Cell 6 – Dimensionality reduction: PCA

from sklearn.decomposition import PCA

# PCA to 2D
pca = PCA(n_components=2, random_state=0)
X_pca = pca.fit_transform(X_array)

print("PCA result shape:", X_pca.shape)
print("PCA explained variance ratio:", pca.explained_variance_ratio_)



PCA result shape: (679968, 2)
PCA explained variance ratio: [0.08834701 0.07278273]


In [ ]:
# Cell 7 – Build 2D embeddings dataframe and save full + sampled CSVs (with work types)

# Start with PCA coordinates and unique ID
embeddings_df = pd.DataFrame({
    "filing_id": df_emb["Job Filing Number"].values,
    "x_pca": X_pca[:, 0],
    "y_pca": X_pca[:, 1],
})

# Base attributes for visualization / linking
attrs_base = [
    "Job Type",
    "Borough",
    "NTA",
    "NTA_coarse",
    "Filing Date",
    "mmYYYY",
    "delta_units",
    "Initial Cost",
    "FloorArea_imputed",
    "approval_days_cap",
    "Latitude",
    "Longitude",
]

# Work-type / flag columns (same as we used in numeric_features)
worktype_cols = [
    "Sprinkler (Work Type)",
    "Plumbing (Work Type)",
    "Standpipe",
    "Antenna",
    "Curb Cut",
    "Sign",
    "Fence",
    "Scaffold",
    "Shed",
    "Boiler Equipment (Work Type)",
    "Earth Work (Work Type)",
    "Foundation (Work Type)",
    "General Construction (Work Type)",
    "Mechanical Systems (Work Type)",
    "Place of Assembly (Work Type)",
    "Protection Mechanical Methods (Work Type)",
    "Sidewalk Shed (Work Type)",
    "Structural (Work Type)",
    "Temporary Place of Assembly (Work Type)",
    "FloorArea_was_imputed",
]

# Optional extra context you might want in tooltips / filters
context_cols = [
    "med_boro",
    "med_boro_job",
    "has_approval_date",
    "has_existing_du",
    "has_proposed_du",
]

attrs = attrs_base + worktype_cols + context_cols

print("Total attrs going into embeddings_df:", len(attrs))

for col in attrs:
    embeddings_df[col] = df_emb[col].values

print("embeddings_df shape:", embeddings_df.shape)
embeddings_df.head()


Total attrs going into embeddings_df: 37
embeddings_df shape: (679968, 40)


,filing_id,x_pca,y_pca,Job Type,Borough,NTA,NTA_coarse,Filing Date,mmYYYY,delta_units,Initial Cost,FloorArea_imputed,approval_days_cap,Latitude,Longitude,Sprinkler (Work Type),Plumbing (Work Type),Standpipe,Antenna,Curb Cut,Sign,Fence,Scaffold,Shed,Boiler Equipment (Work Type),Earth Work (Work Type),Foundation (Work Type),General Construction (Work Type),Mechanical Systems (Work Type),Place of Assembly (Work Type),Protection Mechanical Methods (Work Type),Sidewalk Shed (Work Type),Structural (Work Type),Temporary Place of Assembly (Work Type),FloorArea_was_imputed,med_boro,med_boro_job,has_approval_date,has_existing_du,has_proposed_du
0,B01237951-S1,-0.458456,-1.552765,Alteration,BROOKLYN,Sunset Park (West),Sunset Park (West),2025-08-28,08/2025,0.0,24000.0,600.0,1.0,40.661309,-74.000422,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,2000.0,1435.0,1,1,1
1,B01237951-I1,-0.500723,-0.712571,Alteration,BROOKLYN,Sunset Park (West),Sunset Park (West),2025-06-20,06/2025,0.0,68000.0,1700.0,3.0,40.661309,-74.000422,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,2000.0,1435.0,1,1,1
2,B01258172-I1,1.770822,0.765265,Full Demolition,BROOKLYN,Bedford-Stuyvesant (East),Bedford-Stuyvesant (East),2025-10-12,10/2025,NaN,100000.0,2850.0,NaN,40.685604,-73.926614,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2000.0,2213.0,0,1,0
3,Q01195664-S2,0.532918,0.740239,Alteration,QUEENS,South Richmond Hill,Other NTA,2025-10-12,10/2025,0.0,0.0,625.0,NaN,40.688417,-73.829978,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1611.6,1075.0,0,1,1
4,M01138173-P2,-0.232539,1.717899,Alteration,MANHATTAN,Midtown-Times Square,Midtown-Times Square,2025-10-12,10/2025,0.0,1000.0,1120.0,NaN,40.763527,-73.975279,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1500.0,1460.0,0,1,1


In [ ]:
# Cell 8 – Save full PCA 2D embeddings and sampled subsets

# 1) Full 2D file
full_2d_path = OUTPUT_DIR / "embeddings_2d_full_pca.csv"
embeddings_df.to_csv(full_2d_path, index=False)
print("Saved full 2D embeddings to:", full_2d_path)

# 2) Sampled files: 20k, 30k, 40k, 50k
sample_sizes = [20000, 30000, 40000, 50000]

for n in sample_sizes:
    if embeddings_df.shape[0] >= n:
        sample_df = embeddings_df.sample(n=n, random_state=42)
        out_path = OUTPUT_DIR / f"embeddings_2d_sampled_{n//1000}k.csv"
        sample_df.to_csv(out_path, index=False)
        print(f"Saved sample ({n} rows) to:", out_path)
    else:
        print(f"Requested {n} rows, but only {embeddings_df.shape[0]} available. Skipping {n}.")


Saved full 2D embeddings to: /content/drive/MyDrive/Attempt3-Assignment4-CS424/embeddings_2d_full_pca.csv
Saved sample (20000 rows) to: /content/drive/MyDrive/Attempt3-Assignment4-CS424/embeddings_2d_sampled_20k.csv
Saved sample (30000 rows) to: /content/drive/MyDrive/Attempt3-Assignment4-CS424/embeddings_2d_sampled_30k.csv
Saved sample (40000 rows) to: /content/drive/MyDrive/Attempt3-Assignment4-CS424/embeddings_2d_sampled_40k.csv
Saved sample (50000 rows) to: /content/drive/MyDrive/Attempt3-Assignment4-CS424/embeddings_2d_sampled_50k.csv
